# 05 · Latency percentiles (p50, p95, p99) and cost

Read a latency distribution the way an SRE does: percentiles instead of means, why p95s do not add, why fan-out
makes tails worse, SLOs and error budgets, the TTFT + tokens x TPOT anatomy of an LLM call, the batching trade-off,
and the cost arithmetic from one request to one user-month.

**Time:** ~30 min · **Runs:** offline on CPU in < 30 s · **Needs:** [numpy for AI engineers](../04_numpy_pandas_pytorch/01_numpy_for_ai_engineers.ipynb)

## Why this matters in interviews

- "Your p95 is 6 s and the target is 2 s, what do you do?" is a standard question. It needs you to think in
  percentiles, break latency into stages, and know which levers change which term.
- Means are actively misleading for LLM latency and cost: both are heavy-tailed. Saying "p95" and "p99" unprompted,
  and knowing you cannot add them, is a quick credibility signal.
- Cost questions ("what does this feature cost per month?") are arithmetic, and the candidates who do the arithmetic
  on the whiteboard, including the forgotten tokens, stand out.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `pd41` | What is the difference between p50, p95 and p99, and which do you optimise? |
| `in10` | Your p95 latency is 6 seconds and the target is under 2. What do you do? |
| `in09` | How do you reduce time to first token? |
| `in11` | What is the difference between latency and throughput here, and why do they fight? |
| `in37` | What is tokens-per-second and how do you interpret it? |
| `pd20` | What SLOs would you set for an LLM feature? |
| `in18` | How do you calculate cost per request, properly? |
| `in17` | What are the main cost levers for an LLM application, in order? |
| `pd36` | How do you estimate the cost of an AI feature before building it? |
| `pd26` | What is the cost of a multi-turn conversation, and why does it surprise people? |

## Setup

Most of this notebook is simulation with numpy (deterministic, seeded). One section measures TTFT and time per
output token from a real streamed call through `llm_setup`; offline, the stand-in's `latency` and `token_delay`
knobs play the model.

In [ ]:
import sys
from pathlib import Path
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "llm_setup.py").exists():
        sys.path.insert(0, str(_p)); break
from llm_setup import client, MODEL, EMBED_MODEL, PROVIDER

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
from llm_setup import OFFLINE

rng = np.random.default_rng(0)
client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": "hi"}], max_tokens=5)  # warm-up

def pct(x, q):
    return float(np.percentile(x, q))

## 1. A realistic latency distribution

**In plain English:** most requests take about the same time, but a few take much longer (a retry, a long answer,
a cold replica, a queue). Latency is **right-skewed**: a lognormal body plus a thin tail of slow outliers. We
simulate 10,000 requests of an LLM endpoint.

In [ ]:
n = 10_000
body = rng.lognormal(mean=np.log(1.2), sigma=0.35, size=n)            # typical calls: median ~1.2 s
slow = rng.random(n) < 0.02                                           # 2% hit a retry / timeout / cold start
latency = np.where(slow, body + rng.uniform(4, 12, n), body)

stats = {"mean": latency.mean(), "p50": pct(latency, 50), "p90": pct(latency, 90), "p95": pct(latency, 95),
         "p99": pct(latency, 99), "max": latency.max()}
for k, v in stats.items():
    print(f"{k:>4}: {v:6.2f} s")
assert stats["mean"] > stats["p50"], "a heavy tail drags the mean above the median"
assert stats["p99"] > 3 * stats["p50"]

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.8))
ax.hist(latency, bins=np.logspace(np.log10(0.3), np.log10(20), 80), color="#a5d8ff", edgecolor="#74c0fc")
ax.set_xscale("log")
for k, color in [("p50", "#2f9e44"), ("mean", "#868e96"), ("p95", "#f08c00"), ("p99", "#e03131")]:
    ax.axvline(stats[k], color=color, lw=2, ls="--" if k == "mean" else "-", label=f"{k} = {stats[k]:.2f} s")
ax.set(xlabel="latency (s, log scale)", ylabel="requests", title="10,000 LLM requests: a lognormal body and a slow tail")
ax.legend(); plt.show()

How to read it (`pd41`): half of users wait less than p50; 1 in 20 waits longer than p95; 1 in 100 longer than
p99. The **mean** sits above the median because the tail drags it, so it describes nobody. Optimise p95 for user
experience, watch p99 for the tail that generates tickets, and report TTFT and end-to-end separately.

`np.percentile` interpolates between samples by default. On a dashboard, percentiles usually come from histogram
buckets (Prometheus `histogram_quantile`), which is an estimate too: fine for alerting, but compare like with like.

## 2. You cannot add percentiles

A RAG request runs retrieval, then reranking, then generation. Each stage has its own p95. Is the end-to-end p95
the sum? Only if every stage has its bad day on the same request. For independent stages it is lower; for
correlated ones (a shared overloaded node) it can be anywhere up to the sum.

In [ ]:
stages = {
    "retrieve": rng.lognormal(np.log(0.08), 0.5, n),
    "rerank": rng.lognormal(np.log(0.25), 0.4, n),
    "generate": rng.lognormal(np.log(1.1), 0.35, n),
}
total = sum(stages.values())
sum_of_p95 = sum(pct(v, 95) for v in stages.values())
p95_of_sum = pct(total, 95)
for name, v in stages.items():
    print(f"{name:<9} p50={pct(v, 50):.2f} s  p95={pct(v, 95):.2f} s")
print(f"sum of stage p95s = {sum_of_p95:.2f} s   vs   p95 of the end-to-end time = {p95_of_sum:.2f} s")
assert not np.isclose(sum_of_p95, p95_of_sum, rtol=0.02) and sum_of_p95 > p95_of_sum
print(f"p50s DO NOT add either: sum of p50s {sum(pct(v, 50) for v in stages.values()):.2f} s vs p50 of sum {pct(total, 50):.2f} s")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.2))
left = 0
for (name, v), color in zip(stages.items(), ["#74c0fc", "#ffa94d", "#b197fc"]):
    ax.barh(1, pct(v, 95), left=left, color=color, label=f"{name} p95")
    left += pct(v, 95)
ax.barh(0, p95_of_sum, color="#495057")
ax.set_yticks([0, 1], ["p95 of the\nend-to-end time", "sum of the\nstage p95s"])
ax.set(xlabel="seconds", title="Budgets per stage are fine; adding their p95s is not the end-to-end p95")
ax.legend(loc="lower right", fontsize=8); plt.show()

So: measure the end-to-end percentile directly (a trace per request, see [observability](08_observability_logging_and_tracing.ipynb)),
and use per-stage percentiles to find *where* the time goes. Averaging percentiles across servers is equally
wrong: merge the raw data or the histograms, then take the percentile.

## 3. Fan-out amplifies the tail

A request that calls N services in parallel waits for the **slowest** one. If each call has a 1% chance of being
slower than its p99, the chance that at least one of N is slow is `1 - 0.99^N`: for 10 calls, ~10% of requests hit
somebody's p99; for 100 calls, 63%.

In [ ]:
N = np.arange(1, 101)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for q, color in [(0.95, "#f08c00"), (0.99, "#e03131")]:
    axes[0].plot(N, 1 - q ** N, color=color, lw=2, label=f"one call slower than its p{int(q * 100)}")
axes[0].set(xlabel="parallel calls per request (N)", ylabel="P(request waits for a slow call)",
            title="Probability at least one of N calls is slow")
axes[0].legend()
single = rng.lognormal(np.log(0.2), 0.5, (5_000, 100))                 # 100 parallel calls per request
fan = [1, 2, 5, 10, 20, 50, 100]
axes[1].plot(fan, [pct(single[:, :k].max(axis=1), 50) for k in fan], "o-", label="p50 of the request")
axes[1].plot(fan, [pct(single[:, :k].max(axis=1), 99) for k in fan], "o-", color="#e03131", label="p99 of the request")
axes[1].axhline(pct(single[:, 0], 99), color="k", ls="--", lw=1, label="p99 of ONE call")
axes[1].set_xscale("log")
axes[1].set(xlabel="parallel calls per request (log)", ylabel="request latency (s)", title="Waiting for the slowest of N")
axes[1].legend(fontsize=8); plt.tight_layout(); plt.show()
p50_at_100 = pct(single.max(axis=1), 50)
print(f"with 100 parallel calls, the request's MEDIAN ({p50_at_100:.2f} s) exceeds one call's p99 ({pct(single[:, 0], 99):.2f} s)")
assert abs((1 - 0.99 ** 10) - 0.096) < 0.001 and p50_at_100 > pct(single[:, 0], 99)

This is "The Tail at Scale" (Dean and Barroso) in one chart, and it is why the fixes for fan-out are hedged
requests, per-call timeouts with partial results, fewer and fatter calls, and removing the slowest lanes, not
making the median faster ([async notebook](01_async_concurrency_for_llm_calls.ipynb) has the code).

## 4. SLOs and error budgets

An **SLO** is a promise you can measure: "99% of chat requests get a valid response in under 3 s, over 30 days".
The **error budget** is what the promise leaves: 1% of requests may be slow or failed. Spend it on deploys and
experiments; when it is burning too fast, freeze changes and fix reliability (`pd20`).

In [ ]:
SLO_TARGET, THRESHOLD_S, DAYS, PER_DAY = 0.99, 3.0, 30, 20_000
daily_bad = []
for day in range(DAYS):
    lat = rng.lognormal(np.log(1.2), 0.30, PER_DAY)
    lat = np.where(rng.random(PER_DAY) < 0.002, lat + 5, lat)          # normal days: 0.2% slow
    if day in (11, 12):                                                   # an incident: a bad deploy
        lat = np.where(rng.random(PER_DAY) < 0.06, lat + 5, lat)
    failed = rng.random(PER_DAY) < 0.001
    daily_bad.append(int(((lat > THRESHOLD_S) | failed).sum()))
budget = (1 - SLO_TARGET) * DAYS * PER_DAY
burned = np.cumsum(daily_bad) / budget
print(f"error budget for the month: {budget:,.0f} bad requests out of {DAYS * PER_DAY:,}")
print(f"budget used after day 10: {burned[9]:.0%}, after the incident (day 13): {burned[12]:.0%}, end of month: {burned[-1]:.0%}")
assert burned[12] - burned[10] > 0.15 and burned[-1] < 1.0, "the incident burned a big slice, the month survived"

fig, ax = plt.subplots(figsize=(9, 3.4))
ax.plot(np.arange(1, DAYS + 1), 100 * burned, "o-", ms=3, label="error budget used")
ax.plot([0, DAYS], [0, 100], "k--", lw=1, label="steady burn (exactly on budget)")
ax.axvspan(11.5, 13.5, color="#ffe3e3", label="incident")
ax.set(xlabel="day of the 30-day window", ylabel="% of error budget used", title="SLO: 99% of requests under 3 s")
ax.legend(); plt.show()

A **burn rate** of 1 uses the budget exactly by the end of the window; the incident days burned it several times
faster. Alert on burn rate over two windows (for example "burn rate > 14 over 1 h *and* over 5 min" pages someone;
"> 1 over 3 days" opens a ticket) rather than on single slow requests. For LLM features add a **quality** SLO
(task success on a sampled stream) and a **cost** SLO (p99 cost per request), and decide up front whether a
degraded fallback answer counts as "good".

## 5. Anatomy of one LLM call: TTFT + tokens x TPOT

For a generation call, `total ≈ TTFT + output_tokens x TPOT`:

- **TTFT** (time to first token): network + queueing + **prefill** (reading the whole prompt; grows with prompt
  length; cut by prompt caching and shorter prompts).
- **TPOT** (time per output token, inter-token latency): one **decode** step; memory-bandwidth bound; set by model
  size, hardware and batch load. `1 / TPOT` is the tokens/s a user sees.

Measure both from a streamed call (offline, the stand-in waits 0.25 s, then 20 ms per chunk):

In [ ]:
if OFFLINE:
    OFFLINE.latency, OFFLINE.token_delay = 0.25, 0.02
t0, stamps = time.perf_counter(), []
stream = client.chat.completions.create(model=MODEL, stream=True, max_tokens=80, messages=[{"role": "user", "content":
    "Summarise this.\n\nLatency has two parts. The first token waits for prefill. Every later token costs one decode "
    "step. Long answers are slow because decode is sequential. Short answers are fast."}])
for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content:
        stamps.append(time.perf_counter() - t0)
if OFFLINE:
    OFFLINE.latency, OFFLINE.token_delay = 0.0, 0.0
ttft = stamps[0]
tpot = (stamps[-1] - stamps[0]) / (len(stamps) - 1)
print(f"TTFT = {ttft:.3f} s, TPOT = {1000 * tpot:.1f} ms/token ({1 / tpot:.0f} tokens/s), "
      f"{len(stamps)} chunks, total {stamps[-1]:.2f} s")
print(f"check: TTFT + (n - 1) x TPOT = {ttft + (len(stamps) - 1) * tpot:.2f} s")

In [ ]:
out_tokens = np.arange(0, 1001, 10)
models = {"small model (TTFT 0.3 s, 8 ms/token)": (0.3, 0.008), "large model (TTFT 0.8 s, 25 ms/token)": (0.8, 0.025)}
fig, ax = plt.subplots(figsize=(9, 3.6))
for (name, (t_first, t_tok)), color in zip(models.items(), ["#2f9e44", "#7048e8"]):
    ax.plot(out_tokens, t_first + out_tokens * t_tok, color=color, lw=2, label=name)
    ax.scatter([0], [t_first], color=color, zorder=3)
ax.axhline(2.0, color="#e03131", ls="--", lw=1, label="2 s budget")
ax.set(xlabel="output tokens", ylabel="total latency (s)", title="Output length dominates: total = TTFT + tokens x TPOT")
ax.legend(); plt.show()
budget_tokens = {name: int((2.0 - a) / b) for name, (a, b) in models.items()}
print("output tokens that fit in 2 s:", budget_tokens)

Consequences you can say in one breath (`in10`, `in09`): the cheapest latency fix is usually **fewer output
tokens** (ask for brevity, cap `max_tokens`, structured output instead of prose); TTFT is fixed by **shorter or
cached prompts**, a smaller model, and less queueing; **streaming** changes neither number, only when the user sees
the first word (perceived latency). Humans read roughly 5-10 tokens/s, so anything above ~15 tokens/s per stream
feels fast.

## 6. Throughput vs latency: batching and queueing

A self-hosted GPU decodes many sequences in one step. A decode step is memory-bound (it reads all the weights once
whatever the batch), so a bigger batch costs a little more per step and produces many more tokens. Toy model:
`step_time(b) = 20 ms + 0.25 ms x b`.

In [ ]:
b = np.array([1, 2, 4, 8, 16, 32, 64, 128, 256])
step = 0.020 + 0.00025 * b
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].plot(b, b / step, "o-", color="#2f9e44", label="server throughput (tokens/s, all users)")
axes[0].set_xscale("log", base=2); axes[0].set(xlabel="batch size", ylabel="tokens / s (server)")
ax2 = axes[0].twinx()
ax2.plot(b, 1 / step, "s--", color="#e03131", label="per-user speed (tokens/s)")
ax2.set_ylabel("tokens / s (one user)", color="#e03131")
axes[0].set_title("Batching: throughput up, per-user speed down")
h1, l1 = axes[0].get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
axes[0].legend(h1 + h2, l1 + l2, loc="center left", fontsize=8)

service = 1.0                                           # M/M/1 queue: 1 s mean service time
rho = np.linspace(0.05, 0.97, 60)
arrivals = rng.exponential(1.0, 20_000)
sim_rho, sim_wait = [], []
for r in (0.3, 0.5, 0.7, 0.8, 0.9, 0.95):
    arr_t = np.cumsum(arrivals / r)                    # arrival rate = rho / service
    serv = rng.exponential(service, arrivals.size)
    done = np.zeros_like(arr_t); prev = 0.0
    for i, (a, s) in enumerate(zip(arr_t, serv)):
        prev = max(prev, a) + s; done[i] = prev
    sim_rho.append(r); sim_wait.append(np.mean(done - arr_t))
axes[1].plot(rho, service / (1 - rho), color="#495057", lw=2, label="theory: service / (1 - utilisation)")
axes[1].scatter(sim_rho, sim_wait, color="#e03131", zorder=3, label="simulated queue")
axes[1].set(xlabel="utilisation (arrival rate / capacity)", ylabel="mean time in system (s)",
            title="Queueing: the hockey stick", ylim=(0, 25))
axes[1].legend(fontsize=8); plt.tight_layout(); plt.show()
print(f"batch 1: {1 / step[0]:.0f} tok/s per user, {b[0] / step[0]:.0f} tok/s total | "
      f"batch 256: {1 / step[-1]:.0f} tok/s per user, {b[-1] / step[-1]:,.0f} tok/s total")
assert (b / step)[-1] > 50 * (b / step)[0] and (1 / step)[-1] < (1 / step)[0] / 3

Two lessons (`in11`, `in37`). **Batching** trades per-user speed for total throughput, so serving is a dial, not
an optimum: interactive pools run smaller batches, offline pools run huge ones. And **utilisation** past ~80%
makes queueing delay explode (at 90% the average request spends ten service-times in the system), which is why p99
latency under load is usually queueing, not the model, and why admission control and headroom matter. "3,000
tokens/s" means nothing until someone says at what batch size and whose tokens.

## 7. Cost arithmetic

`cost per call = (uncached_in x p_in + cached_in x p_cached + out x p_out) / 1,000,000`, summed over **every** model
call in the request (main answer, query rewriter, guardrail, judge, retries). Illustrative list prices in USD per 1M
tokens (OpenAI, at the time of writing; always check the current pricing page):

| Model | input | cached input | output |
|---|---|---|---|
| `gpt-4.1` | 2.00 | 0.50 | 8.00 |
| `gpt-4.1-mini` | 0.40 | 0.10 | 1.60 |
| `gpt-4.1-nano` | 0.10 | 0.025 | 0.40 |

Batch APIs are typically about 50% cheaper for work that can wait hours.

In [ ]:
PRICES = {"gpt-4.1": (2.00, 0.50, 8.00), "gpt-4.1-mini": (0.40, 0.10, 1.60), "gpt-4.1-nano": (0.10, 0.025, 0.40)}

def call_cost(model, in_tokens, out_tokens, cached_tokens=0):
    p_in, p_cached, p_out = PRICES[model]
    return ((in_tokens - cached_tokens) * p_in + cached_tokens * p_cached + out_tokens * p_out) / 1e6

def request_cost(calls):
    """calls: every model call behind ONE user request."""
    return sum(call_cost(**c) for c in calls)

# One RAG answer: system prompt + tool schemas + 5 chunks + question; plus a query rewrite and a guardrail check.
rag_request = [
    dict(model="gpt-4.1-mini", in_tokens=900 + 400 + 5 * 350 + 60, out_tokens=250, cached_tokens=1_280),  # answer
    dict(model="gpt-4.1-nano", in_tokens=150, out_tokens=30),                                           # rewrite
    dict(model="gpt-4.1-nano", in_tokens=450, out_tokens=5),                                            # guardrail
]
per_req = request_cost(rag_request)
retry_factor = 1.03                                                     # 3% of calls retried
requests_per_user_day, users = 12, 5_000
print(f"cost per request      : ${per_req:.5f}  (answer call alone: ${call_cost(**rag_request[0]):.5f})")
print(f"per 1M requests       : ${per_req * retry_factor * 1e6:,.0f}")
print(f"per user per month    : ${per_req * retry_factor * requests_per_user_day * 30:.3f}")
print(f"{users:,} users per month : ${per_req * retry_factor * requests_per_user_day * 30 * users:,.0f}")
assert abs(call_cost("gpt-4.1-mini", 1_000_000, 0) - 0.40) < 1e-9

In [ ]:
scenarios = {
    "gpt-4.1, no cache": [dict(rag_request[0], model="gpt-4.1", cached_tokens=0)] + rag_request[1:],
    "gpt-4.1, prompt cache": [dict(rag_request[0], model="gpt-4.1")] + rag_request[1:],
    "4.1-mini, prompt cache": rag_request,
    "4.1-mini + short answers": [dict(rag_request[0], out_tokens=120)] + rag_request[1:],
    "router: 70% nano, 30% mini": None,
}
monthly = {}
for name, calls in scenarios.items():
    if calls is None:
        nano = [dict(rag_request[0], model="gpt-4.1-nano", out_tokens=120)] + rag_request[1:]
        cost = 0.7 * request_cost(nano) + 0.3 * request_cost([dict(rag_request[0], out_tokens=120)] + rag_request[1:])
    else:
        cost = request_cost(calls)
    monthly[name] = cost * retry_factor * requests_per_user_day * 30 * users
fig, ax = plt.subplots(figsize=(9, 3.6))
bars = ax.barh(list(monthly), list(monthly.values()), color="#74c0fc")
ax.bar_label(bars, labels=[f"${v:,.0f}" for v in monthly.values()], padding=3)
ax.invert_yaxis(); ax.set_xlim(0, max(monthly.values()) * 1.25)
ax.set(xlabel="USD per month (5,000 users x 12 requests/day)", title="Same feature, different levers")
plt.show()
assert list(monthly.values()) == sorted(monthly.values(), reverse=True)

Cost levers, roughly in order of return on effort (`in17`; the percentages are typical ranges, not promises):

| Lever | What it cuts | Typical effect | Quality risk |
|---|---|---|---|
| Measure first (cost per feature, per request p50/p99) | finds the one endpoint or loop doing most of the damage | often the biggest win | none |
| Provider prompt caching (stable prefix first) | input cost and TTFT | large on long, repeated prefixes (discounts of 50-90% on the cached part) | none |
| Exact / semantic caching | whole calls | depends on how repetitive traffic is | semantic cache can be wrong |
| Model routing / cascades | price per token for the easy majority | often the biggest structural saving | needs per-route evals |
| Shorter prompts (fewer chunks, tighter system prompt) | input tokens | proportional | measure on evals |
| Shorter outputs (`max_tokens`, brevity, structured output) | output tokens, which cost 3-5x input, and latency | large for chatty features | usually low |
| Batch API for non-interactive work | ~50% | offline jobs only | none |
| Trim conversation history | quadratic growth in long chats | large for long sessions | lose context if done badly |
| Distil or self-host | price per token at high volume | only above a real volume threshold | ops burden |

**Multi-turn conversations surprise people** (`pd26`): a stateless API resends the whole history every turn, so
cumulative input tokens grow with the square of the number of turns.

In [ ]:
def conversation_input_tokens(turns, per_turn=200, system=800):
    return sum(system + per_turn * k for k in range(1, turns + 1))    # turn k resends k turns of history

for turns in (10, 30):
    print(f"{turns} turns: {conversation_input_tokens(turns):,} cumulative input tokens "
          f"(the conversation itself is only {200 * turns:,} tokens)")
assert conversation_input_tokens(30) / conversation_input_tokens(10) > 5

## Try it yourself

**1.** Compute p95 by hand for `[1.1, 0.9, 1.3, 7.5, 1.0, 1.2, 0.8, 1.4, 1.1, 6.0]` with the nearest-rank method
(sort, take the value at rank `ceil(0.95 x n)`), then compare with `np.percentile`. Why do they differ?

In [ ]:
# Solution — try it yourself first, then run this
xs = [1.1, 0.9, 1.3, 7.5, 1.0, 1.2, 0.8, 1.4, 1.1, 6.0]
rank = int(np.ceil(0.95 * len(xs)))
print(f"nearest-rank p95 = {sorted(xs)[rank - 1]}  |  np.percentile (linear interpolation) = {np.percentile(xs, 95):.2f}")
print("With 10 samples 'p95' is basically the maximum: percentiles need many samples to mean anything.")

**2.** Your budget is 2.0 s end to end. Retrieval + rerank take 0.45 s at p95, TTFT is 0.6 s and the model decodes
at 40 ms/token. How many output tokens can you afford, and what `max_tokens` would you set?

In [ ]:
# Solution — try it yourself first, then run this
remaining = 2.0 - 0.45 - 0.6
print(f"{remaining:.2f} s left for decode -> {int(remaining / 0.040)} tokens; set max_tokens a little below, e.g. 20,")
print("or stream so the user reads while it writes, or pick a model with a faster TPOT.")

**3.** A 20-turn support chat with a 1,500-token system prompt and 150 tokens per turn, on `gpt-4.1-mini`, 100 output
tokens per turn. What does one conversation cost with no caching, and with the system prompt cached from turn 2?

In [ ]:
# Solution — try it yourself first, then run this
def chat_cost(turns=20, system=1_500, per_turn=150, out=100, cache_system=False):
    total = 0.0
    for k in range(1, turns + 1):
        in_tok = system + per_turn * k
        cached = (system // 128) * 128 if cache_system and k > 1 else 0
        total += call_cost("gpt-4.1-mini", in_tok, out, cached)
    return total
print(f"no cache: ${chat_cost():.4f} per conversation   system prompt cached: ${chat_cost(cache_system=True):.4f}")

## Say it in an interview

- **Percentiles:** "I optimise p95 and watch p99; the mean hides the tail. Percentiles do not add across stages or
  average across servers: measure end to end from traces and merge histograms." In this notebook the stage p95s
  summed to noticeably more than the end-to-end p95.
- **p95 is 6 s, target 2 s:** trace the stages, attack the biggest term. Usually decode (cap output tokens), then
  prefill (shorter, cached prompt), then serial steps that could run in parallel, then a smaller or routed model;
  stream last, because it changes perception, not the number.
- **Anatomy:** total = TTFT + output tokens x TPOT. TTFT is prompt length, caching and queueing; TPOT is model size,
  hardware and batch load.
- **Throughput vs latency:** batching raises server tokens/s and lowers per-user tokens/s; past ~80% utilisation
  queueing explodes.
- **Cost:** tokens x price over every call in the path, plus retries, reported at p50 and p99 per request, then per
  user-month. Multi-turn chats grow quadratically without trimming or caching.
- **Traps:** averaging percentiles; quoting a vendor's tokens/s without the batch size; forgetting the guardrail and
  rewrite calls; ignoring that output tokens cost several times input.

## Next

- [06 · LiteLLM gateway and model routing](06_litellm_gateway_and_model_routing.ipynb): the routing lever, with cost tracking.
- [08 · Observability](08_observability_logging_and_tracing.ipynb): compute these percentiles from real logs.
- [03 · Caching](03_caching_exact_and_semantic.ipynb) · [KV cache and inference speed](../07_genai_foundations/06_kv_cache_and_inference_speed.ipynb) · [A/B testing and statistics](../12_evaluation/05_ab_testing_and_statistics_for_evals.ipynb).
- Theory: [AI system design concepts](../../ai_system_design_concepts/index.html) (latency budgets with a live waterfall, capacity arithmetic) · [interview bank](../../ai_interview_prep/index.html).